In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.notebook import tqdm
import sys

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent 
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.file_paths import TISSUE_DIMENSIONS
print("Available tissue dims:", TISSUE_DIMENSIONS.keys())

Available tissue dims: dict_keys(['cervix', 'brain', 'afmmm'])


In [3]:
# Add the parent directory to the path to access the utils module
sys.path.append('..')
from src.utils.file_paths import file_paths, TISSUE_DIMENSIONS
from src.utils.pr_test import charpoly, charpoly_vectorized

# Set display options for better visibility
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

## 1. Load and Merge Brain Dataset Files

## 2. Filter samples

In [12]:
def compute_pr_mask_for_file(mueller_matrix, batch_size=50000):
    """
    Compute physical realizability mask for a Mueller matrix array
    
    Args:
        mueller_matrix: Mueller matrix array (H, W, 16)
        batch_size: Number of pixels to process per batch
        
    Returns:
        pr_mask: Boolean mask (H, W) indicating physically realizable pixels
    """
    print(f"Computing PR test for shape {mueller_matrix.shape}...")
    
    H, W, _ = mueller_matrix.shape
    total_pixels = H * W
    
    # Reshape to (N, 4, 4) for batch processing
    mueller_matrices = mueller_matrix.reshape(-1, 16).reshape(-1, 4, 4)
    pr_results = np.zeros(total_pixels, dtype=bool)
    
    # Process in batches using vectorized operations
    for i in range(0, total_pixels, batch_size):
        end_idx = min(i + batch_size, total_pixels)
        batch_matrices = mueller_matrices[i:end_idx]
        
        # Vectorized PR test
        batch_results = charpoly_vectorized(batch_matrices)
        pr_results[i:end_idx] = batch_results
    
    # Reshape back to (H, W)
    pr_mask = pr_results.reshape(H, W)
    pr_pixels = np.sum(pr_mask)
    
    print(f"PR pixels: {pr_pixels:,} ({pr_pixels/total_pixels*100:.2f}%)")
    return pr_mask

In [13]:
def charpoly(M, verbose=False):
    """
    Original charpoly function for single matrix - now uses vectorized version
    
    Args:
        M: Single 4x4 matrix
        verbose: Whether to print coefficient values
        
    Returns:
        Boolean indicating physical realizability
    """
    # Convert single matrix to batch format and use vectorized version
    M_batch = M.reshape(1, 4, 4)
    result = charpoly_vectorized(M_batch, verbose=verbose)
    return result[0]  # Return single boolean value


In [14]:
def compute_ground_truth_pr_mask(mueller_matrix, batch_size=50000):
    """
    Compute the physical realizability mask for a Mueller matrix image.

    Args:
        mueller_matrix: array of shape (H, W, 16)
        batch_size: pixels per batch

    Returns:
        pr_mask: boolean array (H, W)
    """
    H, W, _ = mueller_matrix.shape
    total_pixels = H * W
    mats = mueller_matrix.reshape(-1, 4, 4)
    pr_results = np.zeros(total_pixels, dtype=bool)

    for i in range(0, total_pixels, batch_size):
        end = min(i + batch_size, total_pixels)
        pr_results[i:end] = charpoly_vectorized(mats[i:end])

    return pr_results.reshape(H, W)


In [44]:
def process_raw_to_interim_with_pr(tissue_name, batch_size=50000):
    """
    Process ALL raw files (H,W,16) to interim files (H,W,17) by adding PR mask
    No exclusions at this stage - process everything
    
    Args:
        tissue_name: Name of the tissue type
        batch_size: Batch size for PR computation
    """
    print(f"\n=== Processing {tissue_name.upper()} raw to interim with PR test ===")
    
    # Get paths
    raw_path = getattr(file_paths, f"{tissue_name}_raw_path")
    interim_path = getattr(file_paths, f"{tissue_name}_interim_path")
    
    print(f"Raw path: {raw_path}")
    print(f"Interim path: {interim_path}")
    
    if not raw_path.exists():
        print(f"{tissue_name.capitalize()} raw path does not exist: {raw_path}")
        return
    else:
        print(f"Raw path exists: {raw_path}")
    
    # Create interim directory
    interim_path.mkdir(parents=True, exist_ok=True)
    print(f"Created interim directory: {interim_path}")
    
    # Find ALL .npy files - no exclusions at this stage
    files = list(raw_path.glob('**/*.npy'))
    print(f"Found {len(files)} .npy files in the {tissue_name} raw directory.")
    
    if len(files) == 0:
        print(f"  No .npy files found in {raw_path}")
        print("   Try checking subdirectories or file extensions")
        return
    else:
        print("Processing ALL files (no exclusions at this stage).")
        # Show first few files as examples
        for i, f in enumerate(files[:3]):
            print(f"   Example file {i+1}: {f.name}")
        if len(files) > 3:
            print(f"   ... and {len(files)-3} more files")
    
    # Get expected dimensions
    dims = TISSUE_DIMENSIONS[tissue_name]
    nrows, ncols = dims['num_rows'], dims['num_cols']
    
    # Process each file
    processed_files = []
    for fp in tqdm(files, desc=f"Processing {tissue_name} files with PR test"):
        try:
            # Load raw Mueller matrix (H, W, 16)
            mueller_matrix = np.load(fp)
            
            # Check if it's the expected raw format (H, W, 16)
            if mueller_matrix.shape == (nrows, ncols, 16):
                # Standard format - process as usual
                pr_mask = compute_pr_mask_for_file(mueller_matrix, batch_size)
                combined_array = np.concatenate([mueller_matrix, pr_mask[..., np.newaxis]], axis=2)
                
                # Save to interim with same filename but _combined suffix
                output_name = fp.stem + '_combined.npy'
                output_path = interim_path / output_name
                
                np.save(output_path, combined_array)
                
                processed_files.append({
                    'original_file': str(fp),
                    'interim_file': str(output_path),
                    'shape_original': mueller_matrix.shape,
                    'shape_combined': combined_array.shape,
                    'pr_coverage': np.mean(pr_mask)
                })
                
                print(f"  Processed {fp.name} -> {output_name}")
                print(f"   Saved to: {output_path}")
                print(f"   PR coverage: {np.mean(pr_mask):.3f}")
                
            elif mueller_matrix.shape == (nrows * ncols, 16):
                # Flattened format - need to reshape first
                print(f"Reshaping flattened data {mueller_matrix.shape} -> {(nrows, ncols, 16)}")
                
                # Reshape from (H*W, 16) to (H, W, 16)
                mueller_matrix_reshaped = mueller_matrix.reshape(nrows, ncols, 16)
                
                # Compute PR mask
                pr_mask = compute_pr_mask_for_file(mueller_matrix_reshaped, batch_size)
                
                # Concatenate PR mask as 17th channel
                combined_array = np.concatenate([mueller_matrix_reshaped, pr_mask[..., np.newaxis]], axis=2)
                
                # Save to interim with same filename but _combined suffix
                output_name = fp.stem + '_combined.npy'
                output_path = interim_path / output_name
                
                np.save(output_path, combined_array)
                
                processed_files.append({
                    'original_file': str(fp),
                    'interim_file': str(output_path),
                    'shape_original': mueller_matrix.shape,
                    'shape_combined': combined_array.shape,
                    'pr_coverage': np.mean(pr_mask)
                })
                
                print(f"  Processed {fp.name} -> {output_name} (reshaped)")
                print(f"   Original shape: {mueller_matrix.shape}")
                print(f"   Reshaped to: {mueller_matrix_reshaped.shape}")
                print(f"   Final shape: {combined_array.shape}")
                print(f"   PR coverage: {np.mean(pr_mask):.3f}")
                
            elif mueller_matrix.shape == (nrows, ncols, 17):
                # File already has 17 channels (probably already processed)
                print(f"Skipping {fp.name}: already has 17 channels")
                
            else:
                print(f"Skipping {fp.name}: unexpected shape {mueller_matrix.shape}")
                print(f"Expected: {(nrows, ncols, 16)} or {(nrows * ncols, 16)}")
                
        except Exception as e:
            print(f"Error processing {fp.name}: {e}")
            import traceback
            traceback.print_exc()
    
    # Save processing log
    if processed_files:
        log_df = pd.DataFrame(processed_files)
        log_path = interim_path / f'{tissue_name}_processing_log.csv'
        log_df.to_csv(log_path, index=False)
        print(f"Saved processing log to {log_path}")
    
    print(f"Successfully processed {len(processed_files)} {tissue_name} files to interim.")
    
    # List what files are now in interim directory for verification
    interim_files = list(interim_path.glob('**/*.npy'))
    print(f"Interim directory now contains {len(interim_files)} files:")
    for f in interim_files:
        print(f"   {f.name}")
    
    return len(processed_files)

In [45]:
def merge_interim_files(tissue_name, exclude_names):
    """
    Merge interim files (H,W,17) and save to processed directory
    This is your existing merge logic but adapted for interim files
    """
    print(f"\n=== Merging {tissue_name.upper()} interim files ===")
    
    interim_path = getattr(file_paths, f"{tissue_name}_interim_path")
    processed_path = getattr(file_paths, f"{tissue_name}_processed_path")

    if not interim_path.exists():
        print(f"{tissue_name.capitalize()} interim path does not exist: {interim_path}")
        return
        
    processed_path.mkdir(parents=True, exist_ok=True)

    # Find _combined.npy files in interim
    files = list(interim_path.glob('**/*_combined.npy'))
    print(f"Found {len(files)} combined .npy files in the {tissue_name} interim directory.")

    # Apply exclusions NOW during merging
    exclude_combined = {name.replace('.npy', '_combined.npy') for name in exclude_names}
    files = [fp for fp in files if fp.name not in exclude_combined]
    print(f"{len(files)} files remaining after excluding {len(exclude_combined)} {tissue_name} samples.")

    dims = TISSUE_DIMENSIONS[tissue_name]
    nrows, ncols = dims['num_rows'], dims['num_cols']

    arrays, info = [], []
    for fp in tqdm(files, desc=f"Merging {tissue_name} files"):
        try:
            arr = np.load(fp)
            if arr.shape[:2] == (nrows, ncols) and arr.shape[2] == 17:
                sid = fp.stem.replace('_combined', '')
                arrays.append(arr)
                info.append({
                    'sample_id': sid,
                    'file_path': str(fp),
                    'shape': arr.shape,
                    'mask_coverage': np.mean(arr[..., -1])
                })
            else:
                print(f"Skipping {fp.name}: unexpected shape {arr.shape}")
        except Exception as e:
            print(f"Error loading {fp.name}: {e}")

    if arrays:
        merged = np.stack(arrays, axis=0)
        X = merged[..., :16]  # Mueller matrices
        y = merged[..., -1]   # PR masks

        np.save(processed_path / 'merged_all_X.npy', X)
        np.save(processed_path / 'merged_all_y.npy', y)
        print(f"Saved {tissue_name} merged X (shape {X.shape}) and y (shape {y.shape}).")

        pd.DataFrame(info).to_csv(processed_path / f'{tissue_name}_sample_info.csv', index=False)
        print(f"Saved {tissue_name} sample info CSV.")
        
        return merged, pd.DataFrame(info)
    else:
        print(f"No valid {tissue_name} arrays to merge.")
        return None, pd.DataFrame(info)


In [46]:
# Define exclusion lists
exclude_brain = {
    '2022-02-16_T_HORAO-1-A_FR_15Z_1.npy',  # Remove _combined suffix for raw files
    '2022-03-16_T_HORAO-4-D_FR_1_2.npy',
    '2022-03-03_T_HORAOII-2-C_FR_M_7.npy',
    '2022-02-16_T_HORAO-1-C_FR_15Z_8.npy',
    '2022-02-16_T_HORAO-1-C_FR_15Z_3.npy'
}

exclude_cervix = {
    'Sample2_550_Data.npy',  # Remove _combined suffix for raw files
    'Sample18_550_Data.npy',
    'Sample19_550_Data.npy',
    'Sample23_550_Data.npy',
    'Sample25_550_Data.npy'
}

exclude_afmmm = {
    'AFMMM_sample_he9_Data.npy',  # Remove _combined suffix for raw files
    'AFMMM_sample_he14_Data.npy',
    'AFMMM_sample_bg5_Data.npy',
    'AFMMM_sample_bw6_Data.npy',
    'AFMMM_sample_bg11_Data.npy'
}

In [47]:
# STEP 1: Process raw files to interim with PR test
print("STEP 1: Processing raw files to interim with PR test...")
process_raw_to_interim_with_pr('brain')
process_raw_to_interim_with_pr('cervix')
process_raw_to_interim_with_pr('afmmm')

STEP 1: Processing raw files to interim with PR test...

=== Processing BRAIN raw to interim with PR test ===
Raw path: /Users/chaechae/Desktop/EP_Code/pr_prediction/data/raw/brain
Interim path: /Users/chaechae/Desktop/EP_Code/pr_prediction/data/interim/brain
✅ Raw path exists: /Users/chaechae/Desktop/EP_Code/pr_prediction/data/raw/brain
✅ Created interim directory: /Users/chaechae/Desktop/EP_Code/pr_prediction/data/interim/brain
Found 36 .npy files in the brain raw directory.
Processing ALL files (no exclusions at this stage).
   Example file 1: 2022-03-16_T_HORAO-4-B_FR_1_1.npy
   Example file 2: 2022-03-16_T_HORAO-4-B_FR_1_2.npy
   Example file 3: 2022-03-03_T_HORAOII-2-C_FR_M_8.npy
   ... and 33 more files


Processing brain files with PR test:   0%|          | 0/36 [00:00<?, ?it/s]

Computing PR test for shape (388, 516, 16)...
PR pixels: 183,523 (91.67%)
✅ Processed 2022-03-16_T_HORAO-4-B_FR_1_1.npy -> 2022-03-16_T_HORAO-4-B_FR_1_1_combined.npy
   Saved to: /Users/chaechae/Desktop/EP_Code/pr_prediction/data/interim/brain/2022-03-16_T_HORAO-4-B_FR_1_1_combined.npy
   PR coverage: 0.917
Computing PR test for shape (388, 516, 16)...
PR pixels: 188,585 (94.19%)
✅ Processed 2022-03-16_T_HORAO-4-B_FR_1_2.npy -> 2022-03-16_T_HORAO-4-B_FR_1_2_combined.npy
   Saved to: /Users/chaechae/Desktop/EP_Code/pr_prediction/data/interim/brain/2022-03-16_T_HORAO-4-B_FR_1_2_combined.npy
   PR coverage: 0.942
Computing PR test for shape (388, 516, 16)...
PR pixels: 199,444 (99.62%)
✅ Processed 2022-03-03_T_HORAOII-2-C_FR_M_8.npy -> 2022-03-03_T_HORAOII-2-C_FR_M_8_combined.npy
   Saved to: /Users/chaechae/Desktop/EP_Code/pr_prediction/data/interim/brain/2022-03-03_T_HORAOII-2-C_FR_M_8_combined.npy
   PR coverage: 0.996
Computing PR test for shape (388, 516, 16)...
PR pixels: 200,106 (9

✅ Processed 2022-03-03_T_HORAOII-2-C_FR_M_4.npy -> 2022-03-03_T_HORAOII-2-C_FR_M_4_combined.npy
   Saved to: /Users/chaechae/Desktop/EP_Code/pr_prediction/data/interim/brain/2022-03-03_T_HORAOII-2-C_FR_M_4_combined.npy
   PR coverage: 0.997
Computing PR test for shape (388, 516, 16)...
PR pixels: 178,226 (89.02%)
✅ Processed 2022-03-03_T_HORAOII-2-C_FR_M_5.npy -> 2022-03-03_T_HORAOII-2-C_FR_M_5_combined.npy
   Saved to: /Users/chaechae/Desktop/EP_Code/pr_prediction/data/interim/brain/2022-03-03_T_HORAOII-2-C_FR_M_5_combined.npy
   PR coverage: 0.890
Computing PR test for shape (388, 516, 16)...
PR pixels: 200,192 (99.99%)
✅ Processed 2022-05-04_T_C-EXP3-_FR_M_2.npy -> 2022-05-04_T_C-EXP3-_FR_M_2_combined.npy
   Saved to: /Users/chaechae/Desktop/EP_Code/pr_prediction/data/interim/brain/2022-05-04_T_C-EXP3-_FR_M_2_combined.npy
   PR coverage: 1.000
Computing PR test for shape (388, 516, 16)...
PR pixels: 183,316 (91.56%)
✅ Processed 2022-03-16_T_HORAO-4-A_FR_1_1.npy -> 2022-03-16_T_HORAO

Processing cervix files with PR test:   0%|          | 0/24 [00:00<?, ?it/s]

...

Processing afmmm files with PR test:   0%|          | 0/52 [00:00<?, ?it/s]

🔄 Reshaping flattened data (250000, 16) -> (500, 500, 16)
Computing PR test for shape (500, 500, 16)...
PR pixels: 84,978 (33.99%)
✅ Processed AFMMM_sample_bw1_Data.npy -> AFMMM_sample_bw1_Data_combined.npy (reshaped)
   Original shape: (250000, 16)
   Reshaped to: (500, 500, 16)
   Final shape: (500, 500, 17)
   PR coverage: 0.340
🔄 Reshaping flattened data (250000, 16) -> (500, 500, 16)
Computing PR test for shape (500, 500, 16)...
PR pixels: 65,894 (26.36%)
✅ Processed AFMMM_sample_he1_Data.npy -> AFMMM_sample_he1_Data_combined.npy (reshaped)
   Original shape: (250000, 16)
   Reshaped to: (500, 500, 16)
   Final shape: (500, 500, 17)
   PR coverage: 0.264
🔄 Reshaping flattened data (250000, 16) -> (500, 500, 16)
Computing PR test for shape (500, 500, 16)...
PR pixels: 67,609 (27.04%)
✅ Processed AFMMM_sample_co3_Data.npy -> AFMMM_sample_co3_Data_combined.npy (reshaped)
   Original shape: (250000, 16)
   Reshaped to: (500, 500, 16)
   Final shape: (500, 500, 17)
   PR coverage: 0.27

PR pixels: 57,370 (22.95%)
✅ Processed AFMMM_sample_bg6_Data.npy -> AFMMM_sample_bg6_Data_combined.npy (reshaped)
   Original shape: (250000, 16)
   Reshaped to: (500, 500, 16)
   Final shape: (500, 500, 17)
   PR coverage: 0.229
🔄 Reshaping flattened data (250000, 16) -> (500, 500, 16)
Computing PR test for shape (500, 500, 16)...
PR pixels: 74,684 (29.87%)
✅ Processed AFMMM_sample_he15_Data.npy -> AFMMM_sample_he15_Data_combined.npy (reshaped)
   Original shape: (250000, 16)
   Reshaped to: (500, 500, 16)
   Final shape: (500, 500, 17)
   PR coverage: 0.299
🔄 Reshaping flattened data (250000, 16) -> (500, 500, 16)
Computing PR test for shape (500, 500, 16)...
PR pixels: 64,657 (25.86%)
✅ Processed AFMMM_sample_he14_Data.npy -> AFMMM_sample_he14_Data_combined.npy (reshaped)
   Original shape: (250000, 16)
   Reshaped to: (500, 500, 16)
   Final shape: (500, 500, 17)
   PR coverage: 0.259
🔄 Reshaping flattened data (250000, 16) -> (500, 500, 16)
Computing PR test for shape (500, 500, 1

PR pixels: 75,897 (30.36%)
✅ Processed AFMMM_sample_he13_Data.npy -> AFMMM_sample_he13_Data_combined.npy (reshaped)
   Original shape: (250000, 16)
   Reshaped to: (500, 500, 16)
   Final shape: (500, 500, 17)
   PR coverage: 0.304
Saved processing log to /Users/chaechae/Desktop/EP_Code/pr_prediction/data/interim/afmmm/afmmm_processing_log.csv
Successfully processed 52 afmmm files to interim.
📁 Interim directory now contains 52 files:
   AFMMM_sample_bg2_Data_combined.npy
   AFMMM_sample_bg13_Data_combined.npy
   AFMMM_sample_he13_Data_combined.npy
   AFMMM_sample_he8_Data_combined.npy
   AFMMM_sample_bw8_Data_combined.npy
   AFMMM_sample_bw13_Data_combined.npy
   AFMMM_sample_bg7_Data_combined.npy
   AFMMM_sample_co5_Data_combined.npy
   AFMMM_sample_bg3_Data_combined.npy
   AFMMM_sample_co1_Data_combined.npy
   AFMMM_sample_bg12_Data_combined.npy
   AFMMM_sample_he12_Data_combined.npy
   AFMMM_sample_he9_Data_combined.npy
   AFMMM_sample_bw12_Data_combined.npy
   AFMMM_sample_bw9_Dat

52

In [48]:
# STEP 2: Merge interim files to processed
print("\nSTEP 2: Merging interim files to processed...")
brain_merged, brain_info = merge_interim_files('brain', exclude_brain)
cervix_merged, cervix_info = merge_interim_files('cervix', exclude_cervix)
afmmm_merged, afmmm_info = merge_interim_files('afmmm', exclude_afmmm)


STEP 2: Merging interim files to processed...

=== Merging BRAIN interim files ===
Found 36 combined .npy files in the brain interim directory.
31 files remaining after excluding 5 brain samples.


Merging brain files:   0%|          | 0/31 [00:00<?, ?it/s]

Saved brain merged X (shape (31, 388, 516, 16)) and y (shape (31, 388, 516)).
Saved brain sample info CSV.

=== Merging CERVIX interim files ===
Found 24 combined .npy files in the cervix interim directory.
19 files remaining after excluding 5 cervix samples.


Merging cervix files:   0%|          | 0/19 [00:00<?, ?it/s]

Saved cervix merged X (shape (19, 600, 800, 16)) and y (shape (19, 600, 800)).
Saved cervix sample info CSV.

=== Merging AFMMM interim files ===
Found 52 combined .npy files in the afmmm interim directory.
47 files remaining after excluding 5 afmmm samples.


Merging afmmm files:   0%|          | 0/47 [00:00<?, ?it/s]

Saved afmmm merged X (shape (47, 500, 500, 16)) and y (shape (47, 500, 500)).
Saved afmmm sample info CSV.


In [36]:
# STEP 3: Combine all datasets for ML training
print("\nSTEP 3: Combining all datasets...")

# Check which datasets are available and their dimensions
available_datasets = []
dataset_info = []

for tissue in ['brain', 'cervix', 'afmmm']:
    processed_path = getattr(file_paths, f"{tissue}_processed_path")
    X_file = processed_path / 'merged_all_X.npy'
    y_file = processed_path / 'merged_all_y.npy'
    
    if X_file.exists() and y_file.exists():
        print(f"  {tissue.upper()} dataset files found")
        
        # Load the data to get shape info
        X_data = np.load(X_file)
        y_data = np.load(y_file)
        
        dataset_info.append({
            'tissue': tissue,
            'X_shape': X_data.shape,
            'y_shape': y_data.shape,
            'samples': X_data.shape[0],
            'height': X_data.shape[1],
            'width': X_data.shape[2],
            'channels': X_data.shape[3]
        })
        available_datasets.append(tissue)
        print(f"   X shape: {X_data.shape}, y shape: {y_data.shape}")
        
    else:
        print(f"  {tissue.upper()} dataset files missing:")
        print(f"   X file exists: {X_file.exists()} - {X_file}")
        print(f"   y file exists: {y_file.exists()} - {y_file}")

if not available_datasets:
    print("No processed datasets found! Check Steps 1 and 2.")
    print("Cannot proceed with dataset combination.")
else:
    print(f"\n  Found {len(available_datasets)} datasets: {available_datasets}")
    
    # Check if dimensions are compatible for concatenation
    df_info = pd.DataFrame(dataset_info)
    print("\n📏 Dataset dimensions:")
    print(df_info[['tissue', 'samples', 'height', 'width', 'channels']])
    
    unique_heights = df_info['height'].unique()
    unique_widths = df_info['width'].unique()
    
    if len(unique_heights) == 1 and len(unique_widths) == 1:
        print("  All datasets have compatible dimensions - can concatenate directly")
        
        # Load and combine datasets
        X_arrays = []
        y_arrays = []
        
        for tissue in available_datasets:
            processed_path = getattr(file_paths, f"{tissue}_processed_path")
            X_data = np.load(processed_path / 'merged_all_X.npy')
            y_data = np.load(processed_path / 'merged_all_y.npy')
            X_arrays.append(X_data)
            y_arrays.append(y_data)
            print(f"Loaded {tissue}: X{X_data.shape}, y{y_data.shape}")
        
        # Concatenate along the sample axis
        X_all = np.concatenate(X_arrays, axis=0)
        Y_all = np.concatenate(y_arrays, axis=0)
        
        print(f"\nCombined X_all shape: {X_all.shape}")
        print(f"Combined Y_all shape: {Y_all.shape}")
        
        # Save combined datasets
        combined_path = file_paths.combined_interim_path
        combined_path.mkdir(parents=True, exist_ok=True)
        
        np.save(combined_path / 'X_all_combined.npy', X_all)
        np.save(combined_path / 'Y_all_combined.npy', Y_all)
        
        print(f"Saved combined datasets to {combined_path}")
        
    else:
        print("⚠️  Datasets have incompatible dimensions - cannot concatenate directly")
        print(f"   Heights: {unique_heights}")
        print(f"   Widths: {unique_widths}")
        print("\nOptions:")
        print("1. Use datasets separately for tissue-specific models")
        print("2. Resize all datasets to common dimensions")
        print("3. Use pixel-level flattening approach")
        
        # Save datasets separately but also create flattened version for ML
        combined_path = file_paths.combined_interim_path
        combined_path.mkdir(parents=True, exist_ok=True)
        
        # Option 3: Create flattened pixel-level dataset
        print("\n  Creating flattened pixel-level dataset...")
        
        all_X_pixels = []
        all_y_pixels = []
        all_tissue_labels = []
        
        for i, tissue in enumerate(available_datasets):
            processed_path = getattr(file_paths, f"{tissue}_processed_path")
            X_data = np.load(processed_path / 'merged_all_X.npy')
            y_data = np.load(processed_path / 'merged_all_y.npy')
            
            # Flatten spatial dimensions: (samples, H, W, channels) -> (samples*H*W, channels)
            n_samples, h, w, c = X_data.shape
            X_flat = X_data.reshape(-1, c)  # (samples*H*W, 16)
            y_flat = y_data.reshape(-1)     # (samples*H*W,)
            
            # Create tissue labels
            tissue_labels = np.full(X_flat.shape[0], i)  # 0=brain, 1=cervix, 2=afmmm
            
            all_X_pixels.append(X_flat)
            all_y_pixels.append(y_flat)
            all_tissue_labels.append(tissue_labels)
            
            print(f"   {tissue}: {X_data.shape} -> {X_flat.shape} pixels")
        
        # Combine all pixels
        X_pixels_combined = np.concatenate(all_X_pixels, axis=0)
        y_pixels_combined = np.concatenate(all_y_pixels, axis=0)
        tissue_labels_combined = np.concatenate(all_tissue_labels, axis=0)
        
        print(f"\n  Flattened dataset:")
        print(f"   X_pixels shape: {X_pixels_combined.shape}")
        print(f"   y_pixels shape: {y_pixels_combined.shape}")
        print(f"   tissue_labels shape: {tissue_labels_combined.shape}")
        
        # Save flattened datasets
        np.save(combined_path / 'X_pixels_combined.npy', X_pixels_combined)
        np.save(combined_path / 'y_pixels_combined.npy', y_pixels_combined)
        np.save(combined_path / 'tissue_labels_combined.npy', tissue_labels_combined)
        
        print(f"Saved flattened datasets to {combined_path}")
        
        # Save individual datasets for tissue-specific models
        for tissue in available_datasets:
            tissue_path = combined_path / tissue
            tissue_path.mkdir(exist_ok=True)
            
            processed_path = getattr(file_paths, f"{tissue}_processed_path")
            X_data = np.load(processed_path / 'merged_all_X.npy')
            y_data = np.load(processed_path / 'merged_all_y.npy')
            
            np.save(tissue_path / f'X_{tissue}.npy', X_data)
            np.save(tissue_path / f'y_{tissue}.npy', y_data)
    
    # Save dataset info
    pd.DataFrame(dataset_info).to_csv(combined_path / 'dataset_summary.csv', index=False)
    print("  Complete pipeline finished!")
    
    # Show summary
    total_samples = sum([info['samples'] for info in dataset_info])
    print(f"\n📈 FINAL SUMMARY:")
    print(f"   Datasets processed: {len(available_datasets)}")
    print(f"   Total samples: {total_samples}")
    for info in dataset_info:
        print(f"   {info['tissue']}: {info['samples']} samples, {info['height']}x{info['width']} pixels")
    
    if len(unique_heights) > 1 or len(unique_widths) > 1:
        total_pixels = sum([info['samples'] * info['height'] * info['width'] for info in dataset_info])
        print(f"   Total pixels (flattened): {total_pixels:,}")


STEP 3: Combining all datasets...
✅ BRAIN dataset files found
   X shape: (31, 388, 516, 16), y shape: (31, 388, 516)
✅ CERVIX dataset files found
   X shape: (19, 600, 800, 16), y shape: (19, 600, 800)
✅ AFMMM dataset files found
   X shape: (47, 500, 500, 16), y shape: (47, 500, 500)

📊 Found 3 datasets: ['brain', 'cervix', 'afmmm']

📏 Dataset dimensions:
   tissue  samples  height  width  channels
0   brain       31     388    516        16
1  cervix       19     600    800        16
2   afmmm       47     500    500        16
⚠️  Datasets have incompatible dimensions - cannot concatenate directly
   Heights: [388 600 500]
   Widths: [516 800 500]

Options:
1. Use datasets separately for tissue-specific models
2. Resize all datasets to common dimensions
3. Use pixel-level flattening approach

🔄 Creating flattened pixel-level dataset...
   brain: (31, 388, 516, 16) -> (6206448, 16) pixels
   cervix: (19, 600, 800, 16) -> (9120000, 16) pixels
   afmmm: (47, 500, 500, 16) -> (11750000